### Cálculo de diferencias diarias Rofex

#### Librerías

In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import win32com.client as win32

In [ ]:
import webbrowser

In [ ]:
import sys

In [ ]:
pd.options.display.float_format = lambda x: f'{x:,.2f}'.replace('.', '@').replace(',', '.').replace('@', ',') # Fomato númérico general

In [ ]:
Hoy = pd.Timestamp.today().year *100 + pd.Timestamp.today().month #Define fecha del día.

In [ ]:
Fecha_hoy = pd.Timestamp.today().strftime('%Y-%m-%d')

In [ ]:
Fecha_rofex = pd.Timestamp.today().strftime('%d%m%Y')

In [ ]:
Hoy_b = pd.Timestamp.today()

#### Descarga del archivo de precios del cierre de Rofex

In [ ]:
url = 'https://www.rofex.com.ar/Herramientas/Descargas/New/CierredelDíaA3Mercados' + Fecha_rofex +'.xlsx' # URL del archivo del día

In [ ]:
webbrowser.open(url) # Descarga del archivo de precios del día

#### Control de la descarga

In [ ]:
control = input("¿Se descargó correctamente el archivo? (s/n) ").lower()

In [ ]:
if control in ['s','y']:
    print("✅¡Perfecto! Continuando con la ejecución...")
else:
    print("❌Ejecución abortada por el usuario")
    raise SystemExit("💡Verificar la descarga y ejecutar nuevamente")

#### Lectura de precios del día

In [ ]:
archivo = r'\\ardp.local\uds\group_d\ia214828\Downloads\CierredelDíaA3Mercados' + Fecha_rofex + '.xlsx' # Ruta al archivo descargado

In [ ]:
import_rofex = pd.read_excel(archivo, header=9) # Lectura del archivo.

In [ ]:
import_rofex = import_rofex.iloc[:, [0, 9]]

In [ ]:
import_rofex=import_rofex.rename (columns={'Posición':'CONTRATO','Ajuste':'PRECIO CIERRE'}) #Renombra las columnas.

#### Lectura de archivos

In [ ]:
file = r"R:\Files\BIVSA\ADMOPER\Futuros (ROFEX)\TENENCIAS.xlsx" #Ruta al archivo con las operaciones

In [ ]:
Rofex = pd.read_excel(file,sheet_name="Operado") #Lectura de la tabla de operaciones históricas.

In [ ]:
Cotizaciones=pd.read_excel(r"R:\Files\BIVSA\ADMOPER\Futuros (ROFEX)\Cotizaciones.xlsx") #Lectura de la tabla con el historico de precios

In [ ]:
Cotizaciones['FECHA'] = Cotizaciones['FECHA'].dt.normalize()

#### Tabla de precios del día anterior

In [ ]:
Anteriores=Cotizaciones["FECHA"].dt.date.iloc[-1].strftime('%Y-%m-%d') #Obtiene la última fecha disponible

In [ ]:
Anteriores=Cotizaciones[Cotizaciones["FECHA"]>=Anteriores][["CONTRATO","PRECIO CIERRE"]] #Tabla con los precios del día anterior

In [ ]:
Anteriores=Anteriores.rename (columns={'PRECIO CIERRE':'PRECIO T-1'}) #Renombra las columnas.

#### Agrega precios anteriores a la base de operaciones historicas

In [ ]:
Rofex = Rofex.merge(Anteriores,on = "CONTRATO",how = "left") #Une ambas tablas según el contrato.

In [ ]:
Year = [int(s[-4:]) for s in Rofex ["CONTRATO"].astype(str).values] #Obtiene el año de vto. del contrato.

In [ ]:
Month = [int(s[-6:-4]) for s in Rofex ["CONTRATO"].astype(str).values] #Obtiene el mes de vto. del contrato.

In [ ]:
Rofex["MES V."] = Month #Inserta la columna.

In [ ]:
Rofex["AÑO V."] = Year #Inserta la columna.

In [ ]:
Rofex ["VTO. CONTRATO"] = Rofex ["AÑO V."] *100 + Rofex ["MES V."] #Inserta la columna.

In [ ]:
Rofex=Rofex[Rofex["VTO. CONTRATO"]>=Hoy] #Filtra por los contratos vigentes

In [ ]:
Rofex

### Lectura de precios nuevos

In [ ]:
contratos = Rofex["CONTRATO"].unique() # Crea un array con los contratos sobre los que buscará el precio del día

In [ ]:
datos = {"FECHA":Fecha_hoy,"CONTRATO":contratos}

In [ ]:
datos = pd.DataFrame(datos) #Convierte el diccionario a un DataFrame

In [ ]:
datos = datos.merge(import_rofex,on = "CONTRATO",how = "left") #Une ambas tablas según el contrato.

In [ ]:
datos['FECHA'] = pd.to_datetime(datos['FECHA'])

In [ ]:
Cotizaciones_completos = pd.concat([Cotizaciones, datos], ignore_index=True)

In [ ]:
Cotizaciones_completos['FECHA'] = Cotizaciones_completos['FECHA'].dt.normalize()

In [ ]:
Cotizaciones_completos.to_excel(r"R:\Files\BIVSA\ADMOPER\Futuros (ROFEX)\Cotizaciones.xlsx",index=False) #Exporta la nueva tabla de cotizaciones

### Actualiza la tabla de operaciones con los precios del día

In [ ]:
Precios_dia=Cotizaciones_completos["FECHA"].dt.date.iloc[-1].strftime('%Y-%m-%d') #Obtiene la última fecha disponible

In [ ]:
Nuevos=Cotizaciones_completos[Cotizaciones_completos["FECHA"]>=Precios_dia][["CONTRATO","PRECIO CIERRE"]] #Selecciona en la tabla Cotizaciones los precios del día

In [ ]:
Nuevos=Nuevos.rename (columns={'PRECIO CIERRE':'PRECIO T+0'}) #Renombra las columnas.

In [ ]:
Rofex_diario = Rofex.merge(Nuevos,on = "CONTRATO",how = "left") #Une ambas tablas según el contrato.

In [ ]:
Rofex_diario = Rofex_diario.drop(Rofex_diario.columns[[7,8,9]], axis=1) #Elimina las columnas que no necesita.

In [ ]:
Rofex_diario['PRECIO ANTERIOR']= np.where(Rofex_diario['FECHA'] == Fecha_hoy, Rofex_diario['PRECIO'], Rofex_diario['PRECIO T-1'])

In [ ]:
Rofex_diario["DIFERENCIAS"] = (Rofex_diario["PRECIO T+0"]-Rofex_diario["PRECIO ANTERIOR"])*Rofex_diario["CANTIDAD"]*1000

In [ ]:
Rofex_diario

In [ ]:
Sumarizado=Rofex_diario[['CONTRAPARTE','FONDO','DIFERENCIAS']].groupby(['CONTRAPARTE','FONDO']).sum() # Sumariza las columnas

In [ ]:
Sumarizado['ACCIÓN'] = np.where(Sumarizado['DIFERENCIAS']>0, 'Recibir', np.where(Sumarizado['DIFERENCIAS']<0, 'Transferir',''))

In [ ]:
Sumarizado

In [ ]:
#Envio de e-mail
outlook = win32.Dispatch('outlook.application')
mail = outlook.CreateItem(0)
mail.To= 'globalcustody@icbc.com.ar;rodrigo.cometti@icbc.com.ar;ruben.brunner@icbc.com.ar;rocio.deluciamoreira@icbc.com.ar;santiago.solari@icbc.com.ar;agustina.fernandez@icbc.com.ar;agustina.fernandez@icbc.com.ar'
mail.cc= 'alphafondosdeinversion@icbc.com.ar'
mail.Subject = 'MTM ROFEX ' + Fecha_hoy
mail.HTMLBody = 'Estimados, buenas tardes' + '<br/><br/>'
mail.HTMLBody = mail.HTMLBody + 'Por favor, Transferir/Recibir, según corresponda, en concepto de diferencias diarias:' + '<br/><br/>'
mail.HTMLBody = mail.HTMLBody + '''
                {}'''.format(Sumarizado.to_html(index=True)) +'<br/><br/>'
mail.HTMLBody = mail.HTMLBody + 'Saludos!'
mail.Send()